# HiveMind Benchmark auf Colab A100

Runtime: **A100 GPU** (40 GB reichen für Qwen3.6-35B Q4), High-RAM, ≥100 GB Disk.

Dieses Notebook macht den **echten HiveMind-Harness-Lauf**, nicht den Modell-Benchmark:

1. klont den gepushten `dev`-Branch,
2. installiert HiveMind + llama.cpp CUDA + Qwen3.6-35B + Sharp-Template,
3. startet die HiveMind-Engine auf `:8001`,
4. läuft Smoke (3 Tasks × HiveMind/Aider),
5. optional der volle 21-Task-Lauf,
6. optional 5 DeepSWE-Tasks mit HiveMind und **Modal-Verifier** (kein Docker in Colab).

## Keys

Nur für den **DeepSWE-Modal-Verifier** nötig. Die Zelle mit `MODAL_TOKEN_ID` / `MODAL_TOKEN_SECRET` ist unten. Für den 21-Task-Lauf brauchst du keinen Key.


In [ ]:
# 1) GPU + Disk check
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /root | tail -1


In [ ]:
# 2) Clone the pushed dev branch + bootstrap
![ -d /root/HiveMind_llama/.git ] || git clone --branch dev https://github.com/BoredLuzo/HiveMind_llama.git /root/HiveMind_llama
!cp /root/HiveMind_llama/bench/external/rent_bootstrap.sh /root/rent_bootstrap.sh
!chmod +x /root/rent_bootstrap.sh
!cd /root/HiveMind_llama && git log -1 --format='%h %s'


In [ ]:
# 3) Setup (~15-30 min: Python 3.14, llama.cpp CUDA, 20 GB model, aider)
!bash /root/rent_bootstrap.sh setup


In [ ]:
# 4) Start the HiveMind engine in the background
!bash /root/rent_bootstrap.sh serve
!curl -fsS http://127.0.0.1:8001/health


In [ ]:
# 5) Smoke: 3 tasks x 2 conditions (HiveMind, Aider)
# READ the output: pass/fail, tokens > 0, wall time, no 'aider is not installed'.
!bash /root/rent_bootstrap.sh smoke


In [ ]:
# 6) Optional: full 21-task run in tmux
import os
os.environ['HM_RUNS'] = '1'
os.environ['HM_CONDITIONS'] = 'hivemind_duo,aider'
os.environ['HM_TASKS'] = 'bench/tasks'
!bash /root/rent_bootstrap.sh full


In [ ]:
# 7) Progress (re-run as often as you like)
!bash /root/rent_bootstrap.sh status


## DeepSWE-5 mit HiveMind (echter Harness, Modal-Verifier)

Nur nötig, wenn du zusätzlich den externen DeepSWE-Anker willst. HiveMind läuft weiterhin über `/stream`; nur der Verifier läuft auf Modal, weil Colab kein Docker mag.

Trage unten deinen Modal-Token ein. Wenn du das Notebook teilst, vorher löschen/rotieren.


In [ ]:
# 8) Modal token — HIER eintragen (nur für DeepSWE nötig)
MODAL_TOKEN_ID = ""       # <<< HIER Token ID
MODAL_TOKEN_SECRET = ""   # <<< HIER Token Secret

import os
if not MODAL_TOKEN_ID or not MODAL_TOKEN_SECRET:
    print("DeepSWE-Zelle wird ohne Modal-Token nicht laufen.")
else:
    os.environ["MODAL_TOKEN_ID"] = MODAL_TOKEN_ID
    os.environ["MODAL_TOKEN_SECRET"] = MODAL_TOKEN_SECRET
    print("Modal-Token gesetzt:", MODAL_TOKEN_ID[:8] + "...")


In [ ]:
# 9) DeepSWE tasks + Modal SDK im HiveMind-venv
![ -d /root/deep-swe/.git ] || git clone --depth 1 https://github.com/datacurve-ai/deep-swe.git /root/deep-swe
!cd /root/HiveMind_llama && (/root/.local/bin/uv pip install --python .venv/bin/python -q modal || uv pip install --python .venv/bin/python -q modal)


In [ ]:
# 10) HiveMind on 5 DeepSWE tasks, verifier on Modal (no local Docker)
!cd /root/HiveMind_llama && .venv/bin/python bench/external/deepswe_hivemind.py   --tasks /root/deep-swe/tasks   --hivemind http://127.0.0.1:8001   --model qwen3.6:35b-a3b-ud   --n-tasks 5 --sample-seed 0   --verifier modal   --out /root/HiveMind_llama/bench/results/deepswe-hivemind-5


In [ ]:
# 11) Save results to Drive (run after smoke/full/DeepSWE)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/hivemind_bench
!cp -v /root/HiveMind_llama/bench/results/*.json /content/drive/MyDrive/hivemind_bench/
!cp -v /root/HiveMind_llama/bench_full.log /content/drive/MyDrive/hivemind_bench/ 2>/dev/null || true


In [ ]:
# 12) Stop the engine; then terminate the Colab runtime / rental instance.
!bash /root/rent_bootstrap.sh stop
